<a href="https://colab.research.google.com/github/tanusreemajumder/Neural-Networks-Andrew-Ng/blob/main/part_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(1)
print("NumPy version:", np.__version__)

NumPy version: 2.1.3


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [4]:
df = pd.read_csv('/content/drive/MyDrive/Housing.csv')

In [5]:
median_price = df["price"].median()
df["expensive"] = (df["price"] > median_price).astype(int)

yes_no_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]
for col in yes_no_cols:
    df[col] = (df[col] == "yes").astype(int)

df = pd.get_dummies(df, columns=["furnishingstatus"], drop_first=True, dtype=int)

feature_cols = [c for c in df.columns if c not in ("price", "expensive")]
X_all = df[feature_cols].values.astype(float)
Y_all = df["expensive"].values.reshape(-1, 1).astype(float)
n_x = X_all.shape[1]

print("n_x:", n_x, " total examples:", X_all.shape[0])

n_x: 13  total examples: 545


In [6]:
def sigmoid(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))   # clip avoids exp() overflow
def relu(z): return np.maximum(0, z)
def relu_derivative(z): return (z > 0).astype(float)

def compute_cost(AL, Y, eps=1e-8):
    m = Y.shape[1]
    ALc = np.clip(AL, eps, 1 - eps)
    return np.squeeze(-(1.0 / m) * np.sum(Y * np.log(ALc) + (1 - Y) * np.log(1 - ALc)))

def initialize_parameters_deep(layer_dims, seed=1):
    rng = np.random.RandomState(seed)
    parameters = {}
    L = len(layer_dims) - 1
    for l in range(1, L + 1):
        parameters[f"W{l}"] = rng.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2.0 / layer_dims[l-1])
        parameters[f"b{l}"] = np.zeros((layer_dims[l], 1))
    return parameters

def L_model_forward(X, parameters):
    caches = []
    A = X
    L = len(parameters) // 2
    for l in range(1, L):
        A_prev = A
        Z = np.dot(parameters[f"W{l}"], A_prev) + parameters[f"b{l}"]
        A = relu(Z)
        caches.append((A_prev, parameters[f"W{l}"], parameters[f"b{l}"], Z))
    ZL = np.dot(parameters[f"W{L}"], A) + parameters[f"b{L}"]
    AL = sigmoid(ZL)
    caches.append((A, parameters[f"W{L}"], parameters[f"b{L}"], ZL))
    return AL, caches

def L_model_backward(AL, Y, caches):
    grads = {}
    L = len(caches)
    Y = Y.reshape(AL.shape)
    dAL = -(np.divide(Y, np.clip(AL, 1e-8, 1)) - np.divide(1 - Y, np.clip(1 - AL, 1e-8, 1)))
    A_prev, W, b, Z = caches[L - 1]
    m = A_prev.shape[1]
    dZ = dAL * (AL * (1 - AL))
    grads[f"dW{L}"] = (1.0/m) * np.dot(dZ, A_prev.T)
    grads[f"db{L}"] = (1.0/m) * np.sum(dZ, axis=1, keepdims=True)
    dA = np.dot(W.T, dZ)
    for l in reversed(range(1, L)):
        A_prev, W, b, Z = caches[l - 1]
        m = A_prev.shape[1]
        dZ = dA * relu_derivative(Z)
        grads[f"dW{l}"] = (1.0/m) * np.dot(dZ, A_prev.T)
        grads[f"db{l}"] = (1.0/m) * np.sum(dZ, axis=1, keepdims=True)
        dA = np.dot(W.T, dZ)
    return grads

def update_parameters_gd(parameters, grads, alpha):
    L = len(parameters) // 2
    for l in range(1, L + 1):
        parameters[f"W{l}"] -= alpha * grads[f"dW{l}"]
        parameters[f"b{l}"] -= alpha * grads[f"db{l}"]
    return parameters

def predict_deep(parameters, X, threshold=0.5):
    AL, _ = L_model_forward(X, parameters)
    return (AL > threshold).astype(int)

def train_basic(X, Y, layer_dims, num_iterations=1000, alpha=0.2, seed=1):
    params = initialize_parameters_deep(layer_dims, seed=seed)
    costs = []
    for it in range(num_iterations):
        AL, caches = L_model_forward(X, params)
        costs.append(compute_cost(AL, Y))
        grads = L_model_backward(AL, Y, caches)
        params = update_parameters_gd(params, grads, alpha)
    return params, costs

def split_and_normalize(X_all, Y_all, train_frac=0.6, dev_frac=0.2, seed=42):
    m_total = X_all.shape[0]
    perm = np.random.RandomState(seed).permutation(m_total)
    n_train, n_dev = int(train_frac * m_total), int(dev_frac * m_total)
    tr, dv, te = perm[:n_train], perm[n_train:n_train+n_dev], perm[n_train+n_dev:]
    mu = X_all[tr].mean(axis=0, keepdims=True)
    sigma = X_all[tr].std(axis=0, keepdims=True) + 1e-8
    X_train = ((X_all[tr] - mu) / sigma).T
    X_dev   = ((X_all[dv] - mu) / sigma).T
    X_test  = ((X_all[te] - mu) / sigma).T
    return X_train, Y_all[tr].T, X_dev, Y_all[dv].T, X_test, Y_all[te].T, tr, dv, te

X_train, Y_train, X_dev, Y_dev, X_test, Y_test, tr_idx, dv_idx, te_idx = split_and_normalize(X_all, Y_all)
print("Train:", X_train.shape, " Dev:", X_dev.shape, " Test:", X_test.shape)

Train: (13, 327)  Dev: (13, 109)  Test: (13, 109)


## 1. Cleaning Up Incorrectly Labelled Data

Real datasets often have some wrong labels. Deep learning is surprisingly **robust to random label noise** in the training set, as long as there's enough data and the errors really are random (not systematic). Dev/test set label errors matter more directly, since they change your *measurement* of how well the model is doing — worth manually reviewing during error analysis (extend the error-analysis table from the earlier notebook with an "incorrect label" column).

Below: two experiments. First, how much does **random** label noise in training hurt, at increasing rates? Second — the more important distinction — does **systematic** noise (wrong in a consistent, biased way) hurt more than random noise at the same overall rate?

In [7]:
def inject_random_noise(Y, noise_rate, seed=0):
    rng = np.random.RandomState(seed)
    Y_noisy = Y.copy()
    m = Y.shape[1]
    flip_mask = rng.rand(m) < noise_rate
    Y_noisy[0, flip_mask] = 1 - Y_noisy[0, flip_mask]
    return Y_noisy

dims_noise = [n_x, 8, 4, 1]
print(f"{'random noise rate':>18} | {'dev accuracy (clean dev set)':>29}")
print("-" * 52)
for rate in [0.0, 0.05, 0.15, 0.30]:
    Y_train_noisy = inject_random_noise(Y_train, rate)
    params_n, _ = train_basic(X_train, Y_train_noisy, dims_noise, num_iterations=1000, alpha=0.2)
    dev_acc = np.mean(predict_deep(params_n, X_dev) == Y_dev) * 100   # dev set stays CLEAN
    print(f"{rate*100:>17.0f}% | {dev_acc:>28.1f}%")

print("\n-> Accuracy degrades gradually, not catastrophically, as RANDOM label noise increases.")

 random noise rate |  dev accuracy (clean dev set)
----------------------------------------------------
                0% |                         82.6%
                5% |                         77.1%
               15% |                         72.5%
               30% |                         63.3%

-> Accuracy degrades gradually, not catastrophically, as RANDOM label noise increases.


In [9]:
def inject_systematic_noise(Y, X_raw_col, threshold, noise_rate, seed=0):
    """Flip labels ONLY for examples on one side of a feature threshold -- a biased, non-random error."""
    rng = np.random.RandomState(seed)
    Y_noisy = Y.copy()
    m = Y.shape[1]
    eligible = (X_raw_col < threshold)   # e.g. systematically mislabel smaller/cheaper-looking houses
    flip_mask = eligible & (rng.rand(m) < noise_rate / eligible.mean())  # scale so overall rate matches
    Y_noisy[0, flip_mask] = 1 - Y_noisy[0, flip_mask]
    actual_rate = flip_mask.mean()
    return Y_noisy, actual_rate

area_train_raw = X_all[tr_idx, feature_cols.index("area")]
area_median = np.median(area_train_raw)

Y_train_systematic, actual_rate = inject_systematic_noise(Y_train, area_train_raw, area_median, noise_rate=0.15)
Y_train_random = inject_random_noise(Y_train, actual_rate)   # SAME overall noise rate, but random

params_sys, _  = train_basic(X_train, Y_train_systematic, dims_noise, num_iterations=1000, alpha=0.2)
params_rand, _ = train_basic(X_train, Y_train_random,     dims_noise, num_iterations=1000, alpha=0.2)

# The key question isn't overall dev accuracy -- it's whether accuracy specifically COLLAPSES in the
# region that was systematically mislabeled (small-area houses), which random noise wouldn't cause.
area_dev_raw = X_all[dv_idx, feature_cols.index("area")]
small_area_mask = area_dev_raw < area_median

print(f"Actual noise rate injected: {actual_rate*100:.1f}% (same for both experiments below)\n")
print(f"{'':>12} | {'small-area dev acc':>19} | {'large-area dev acc':>19}")
print("-" * 58)
for name, params in [("systematic", params_sys), ("random", params_rand)]:
    preds = predict_deep(params, X_dev).ravel()
    true = Y_dev.ravel()
    acc_small = np.mean(preds[small_area_mask] == true[small_area_mask]) * 100
    acc_large = np.mean(preds[~small_area_mask] == true[~small_area_mask]) * 100
    print(f"{name:>12} | {acc_small:>18.1f}% | {acc_large:>18.1f}%")

print("\n-> Systematic noise causes accuracy to COLLAPSE specifically in the corrupted region (small-area")
print("   houses), while the untouched region stays fine -- the model learned a genuinely wrong, biased")
print("   pattern there. Random noise spreads confusion roughly evenly across both regions instead.")

Actual noise rate injected: 18.3% (same for both experiments below)

             |  small-area dev acc |  large-area dev acc
----------------------------------------------------------
  systematic |               68.9% |               87.5%
      random |               75.4% |               64.6%

-> Systematic noise causes accuracy to COLLAPSE specifically in the corrupted region (small-area
   houses), while the untouched region stays fine -- the model learned a genuinely wrong, biased
   pattern there. Random noise spreads confusion roughly evenly across both regions instead.


## 2. Build First System Quickly, Then Iterate

Especially for a new problem, don't over-plan the "perfect" architecture up front. Build a simple baseline fast (a small network, or even plain logistic regression), get it running end-to-end (data -> train -> evaluate), and use that working system's **bias/variance analysis and error analysis** (both covered in the earlier notebook) to decide what's actually worth improving next — rather than guessing.

This mirrors exactly what these notebooks have been doing all along: logistic regression first, then a shallow network, then deeper, then regularized, then better-optimized — each step motivated by a concrete diagnosis from the previous one, not by starting with the most complex architecture available.

In [10]:
# The "v1" quick baseline (reusing the very first model from these notebooks)
params_v1, costs_v1 = train_basic(X_train, Y_train, [n_x, 1], num_iterations=500, alpha=0.5)  # plain logistic regression
dev_acc_v1 = np.mean(predict_deep(params_v1, X_dev) == Y_dev) * 100
train_acc_v1 = np.mean(predict_deep(params_v1, X_train) == Y_train) * 100
print(f"v1 (logistic regression, built in minutes): train acc = {train_acc_v1:.1f}%, dev acc = {dev_acc_v1:.1f}%")

gap = train_acc_v1 - dev_acc_v1
if train_acc_v1 < 85:
    print("-> Diagnosis: meaningful room to improve TRAINING fit -> try v2 with more capacity (a hidden layer)")
elif abs(gap) > 8:
    print("-> Diagnosis: train/dev gap is large -> try v2 with regularization or more data")
else:
    print("-> Diagnosis: v1 is already fairly balanced -> error analysis to find the next specific improvement")

v1 (logistic regression, built in minutes): train acc = 82.0%, dev acc = 86.2%
-> Diagnosis: meaningful room to improve TRAINING fit -> try v2 with more capacity (a hidden layer)


## 3. Training and Testing on Different Distributions

Sometimes there's abundant data from one distribution but only limited data from the distribution you actually care about. Example here: suppose `prefarea` houses (in the "preferred area") represent the real target market, but most of the dataset is non-`prefarea` houses.

**The recommended split:** train on a mix dominated by the plentiful (non-target) distribution, but make dev and test **purely** from the target distribution — never randomly mix everything together, or you'd be measuring progress on a blend that doesn't match what you actually need to deploy against.

In [11]:
prefarea_col = feature_cols.index("prefarea")
is_prefarea = X_all[:, prefarea_col] == 1

print(f"prefarea houses (target distribution): {is_prefarea.sum()} of {len(is_prefarea)}")
print(f"non-prefarea houses (plentiful, different distribution): {(~is_prefarea).sum()}")

rng = np.random.RandomState(7)
prefarea_idx = np.where(is_prefarea)[0]
nonprefarea_idx = np.where(~is_prefarea)[0]
rng.shuffle(prefarea_idx); rng.shuffle(nonprefarea_idx)

# Dev/test: ONLY prefarea (target distribution), split in half
n_pref_dev = len(prefarea_idx) // 2
dev_idx_mismatch  = prefarea_idx[:n_pref_dev]
test_idx_mismatch = prefarea_idx[n_pref_dev:]

# Train: the rest of prefarea PLUS all of non-prefarea (plentiful, different distribution)
train_idx_mismatch = np.concatenate([nonprefarea_idx])   # (leaving prefarea entirely for dev/test here,
                                                            #  the extreme version of this scenario)

mu = X_all[train_idx_mismatch].mean(axis=0, keepdims=True)
sigma = X_all[train_idx_mismatch].std(axis=0, keepdims=True) + 1e-8
X_train_mm = ((X_all[train_idx_mismatch] - mu) / sigma).T
X_dev_mm   = ((X_all[dev_idx_mismatch]   - mu) / sigma).T
X_test_mm  = ((X_all[test_idx_mismatch]  - mu) / sigma).T
Y_train_mm = Y_all[train_idx_mismatch].T
Y_dev_mm   = Y_all[dev_idx_mismatch].T
Y_test_mm  = Y_all[test_idx_mismatch].T

print(f"\nTrain: {X_train_mm.shape[1]} examples (non-prefarea)")
print(f"Dev:   {X_dev_mm.shape[1]} examples (prefarea -- the TARGET distribution)")
print(f"Test:  {X_test_mm.shape[1]} examples (prefarea -- the TARGET distribution)")

prefarea houses (target distribution): 128 of 545
non-prefarea houses (plentiful, different distribution): 417

Train: 417 examples (non-prefarea)
Dev:   64 examples (prefarea -- the TARGET distribution)
Test:  64 examples (prefarea -- the TARGET distribution)


## 4. Bias and Variance With Mismatched Data

With matched train/dev distributions, a train-dev gap always meant "variance." With **mismatched** distributions, that's no longer clear — a gap could mean the model overfit, **or** it could simply mean dev examples are inherently different/harder. To tell these apart, carve out a **train-dev set**: a held-out slice from the *training* distribution (same distribution as train, just not trained on).

Four numbers now separate cleanly into three gaps:

$$\text{avoidable bias} = \text{train error} - \text{human-level error}$$
$$\text{variance} = \text{train-dev error} - \text{train error} \quad \text{(same distribution, unseen data)}$$
$$\text{data mismatch} = \text{dev error} - \text{train-dev error} \quad \text{(different distribution)}$$

In [12]:
# Carve a train-dev set OUT of the training distribution (non-prefarea), same distribution as train
rng2 = np.random.RandomState(3)
perm_train = rng2.permutation(len(train_idx_mismatch))
n_traindev = int(0.2 * len(train_idx_mismatch))
traindev_local_idx = perm_train[:n_traindev]
actual_train_local_idx = perm_train[n_traindev:]

X_train_final   = X_train_mm[:, actual_train_local_idx]
Y_train_final   = Y_train_mm[:, actual_train_local_idx]
X_traindev      = X_train_mm[:, traindev_local_idx]
Y_traindev      = Y_train_mm[:, traindev_local_idx]

dims_mm = [n_x, 8, 4, 1]
params_mm, _ = train_basic(X_train_final, Y_train_final, dims_mm, num_iterations=1500, alpha=0.2)

train_err     = 100 - np.mean(predict_deep(params_mm, X_train_final) == Y_train_final) * 100
traindev_err  = 100 - np.mean(predict_deep(params_mm, X_traindev)    == Y_traindev)    * 100
dev_err       = 100 - np.mean(predict_deep(params_mm, X_dev_mm)      == Y_dev_mm)      * 100

print(f"Train error:      {train_err:.1f}%")
print(f"Train-dev error:  {traindev_err:.1f}%   (same distribution as train, unseen)")
print(f"Dev error:        {dev_err:.1f}%   (DIFFERENT distribution -- prefarea houses)")

variance = traindev_err - train_err
data_mismatch = dev_err - traindev_err
print(f"\nVariance (train-dev minus train):     {variance:+.1f} pts")
print(f"Data mismatch (dev minus train-dev):  {data_mismatch:+.1f} pts")

if abs(data_mismatch) > abs(variance):
    print("\n-> The bigger gap is DATA MISMATCH, not variance -- the fix is making training data more")
    print("   similar to the target (prefarea) distribution, not more regularization or more train data.")
else:
    print("\n-> The bigger gap is VARIANCE -- the usual regularization/more-data fixes apply here instead.")

Train error:      8.4%
Train-dev error:  22.9%   (same distribution as train, unseen)
Dev error:        79.7%   (DIFFERENT distribution -- prefarea houses)

Variance (train-dev minus train):     +14.5 pts
Data mismatch (dev minus train-dev):  +56.8 pts

-> The bigger gap is DATA MISMATCH, not variance -- the fix is making training data more
   similar to the target (prefarea) distribution, not more regularization or more train data.


## 5. Addressing Data Mismatch

Once data mismatch (not variance) is identified as the real gap, the fixes are different from the usual variance playbook:

- **Manually study the difference** between training and target-distribution examples — what's systematically different? (In a real vision/speech project this might be background noise, lighting, accent, etc.; here it's whatever distinguishes `prefarea` from non-`prefarea` houses.)
- **Make training data more similar to the target distribution** — collect more target-distribution data if possible, or, as a cheaper substitute, apply **artificial data synthesis**: transform existing training examples to look statistically more like the target distribution.

**A well-known caveat with synthesis:** it only helps if it preserves a genuinely correct input-output relationship for the synthesized examples (e.g. adding real recorded car noise to clean speech audio, so a "noisy speech -> correct transcript" pair is still accurate) — not just making inputs superficially resemble the target statistically. Below: a naive synthesis attempt that runs into exactly this problem, to make the caveat concrete rather than abstract.

In [13]:
# Study the difference first: which features differ most between the two distributions?
mu_pref = X_all[is_prefarea].mean(axis=0)
mu_nonpref = X_all[~is_prefarea].mean(axis=0)
diff_table = pd.DataFrame({
    "feature": feature_cols,
    "prefarea mean": mu_pref,
    "non-prefarea mean": mu_nonpref,
    "difference": mu_pref - mu_nonpref,
}).sort_values("difference", key=abs, ascending=False)
print("Where the two distributions differ most:")
print(diff_table.head(5).to_string(index=False))

Where the two distributions differ most:
 feature  prefarea mean  non-prefarea mean  difference
    area    6069.320312        4868.517986 1200.802327
prefarea       1.000000           0.000000    1.000000
basement       0.546875           0.290168    0.256707
 parking       0.835938           0.649880    0.186057
mainroad       0.984375           0.820144    0.164231


In [15]:
# Artificial synthesis: shift each non-prefarea training example's features partway toward the
# prefarea distribution's mean (a crude stand-in for more sophisticated data-synthesis techniques)
shift_fraction = 0.4
X_train_synth_raw = X_all[train_idx_mismatch][actual_train_local_idx].copy()
X_train_synth_raw = X_train_synth_raw + shift_fraction * (mu_pref - mu_nonpref)

X_train_synth = ((X_train_synth_raw - mu) / sigma).T   # same train-set mu/sigma as before, for a fair comparison
Y_train_synth = Y_train_final

params_synth, _ = train_basic(X_train_synth, Y_train_synth, dims_mm, num_iterations=1500, alpha=0.2)
dev_err_synth = 100 - np.mean(predict_deep(params_synth, X_dev_mm) == Y_dev_mm) * 100

print(f"Dev error WITHOUT synthesis: {dev_err:.1f}%")
print(f"Dev error WITH synthesis:    {dev_err_synth:.1f}%")
print("\nThe naive shift barely moves the number -- and that's actually an important lesson, not a")
print("failed experiment. We shifted the X features to LOOK like the target distribution, but kept the")
print("original Y labels (which were correct for the ORIGINAL, unshifted houses). That mismatched the")
print("X-Y relationship itself, so the network learned little useful from the synthesized examples.")
print("\nThis is exactly the real risk with artificial data synthesis that the strategy warns about:")
print("synthesized examples must preserve a genuinely correct X-Y relationship (like adding real car")
print("noise to clean audio for a speech model), not just superficially resemble the target distribution")
print("statistically -- otherwise the model may learn to fit synthesis artifacts instead of anything that")
print("transfers to real target-distribution data.")

Dev error WITHOUT synthesis: 79.7%
Dev error WITH synthesis:    79.7%

The naive shift barely moves the number -- and that's actually an important lesson, not a
failed experiment. We shifted the X features to LOOK like the target distribution, but kept the
original Y labels (which were correct for the ORIGINAL, unshifted houses). That mismatched the
X-Y relationship itself, so the network learned little useful from the synthesized examples.

This is exactly the real risk with artificial data synthesis that the strategy warns about:
synthesized examples must preserve a genuinely correct X-Y relationship (like adding real car
noise to clean audio for a speech model), not just superficially resemble the target distribution
statistically -- otherwise the model may learn to fit synthesis artifacts instead of anything that
transfers to real target-distribution data.


## 6. Transfer Learning

Transfer learning reuses a network trained on task A (usually with **lots** of data) as a starting point for task B (usually with **less** data), when both tasks share useful low-level structure. Typically: keep the early layers' learned weights (**freeze** them), replace/retrain the final layer(s) for the new task (**fine-tune**).

Here: pre-train on the (plentiful-label) 3-class softmax price task from an earlier notebook, then transfer those learned early-layer weights to a **different**, data-scarce downstream task — predicting whether a house has `airconditioning`, using only a small subset of examples — and compare against training that downstream task completely from scratch.

In [16]:
# Softmax helpers (from the earlier batch-norm/softmax notebook)
def softmax(Z):
    Zs = Z - np.max(Z, axis=0, keepdims=True)
    eZ = np.exp(Zs)
    return eZ / np.sum(eZ, axis=0, keepdims=True)

def one_hot(Y_class, num_classes):
    m = Y_class.shape[0]
    Y_oh = np.zeros((num_classes, m))
    Y_oh[Y_class, np.arange(m)] = 1
    return Y_oh

def L_model_forward_softmax(X, parameters):
    caches = []
    A = X
    L = len(parameters) // 2
    for l in range(1, L):
        A_prev = A
        Z = np.dot(parameters[f"W{l}"], A_prev) + parameters[f"b{l}"]
        A = relu(Z)
        caches.append((A_prev, parameters[f"W{l}"], parameters[f"b{l}"], Z))
    ZL = np.dot(parameters[f"W{L}"], A) + parameters[f"b{L}"]
    AL = softmax(ZL)
    caches.append((A, parameters[f"W{L}"], parameters[f"b{L}"], ZL))
    return AL, caches

def softmax_cost(AL, Y_onehot, eps=1e-8):
    m = Y_onehot.shape[1]
    return -np.sum(Y_onehot * np.log(np.clip(AL, eps, 1))) / m

def L_model_backward_softmax(AL, Y_onehot, caches):
    grads = {}
    L = len(caches)
    A_prev, W, b, Z = caches[L - 1]
    m = A_prev.shape[1]
    dZ = AL - Y_onehot
    grads[f"dW{L}"] = (1.0/m) * np.dot(dZ, A_prev.T)
    grads[f"db{L}"] = (1.0/m) * np.sum(dZ, axis=1, keepdims=True)
    dA = np.dot(W.T, dZ)
    for l in reversed(range(1, L)):
        A_prev, W, b, Z = caches[l - 1]
        m = A_prev.shape[1]
        dZ = dA * relu_derivative(Z)
        grads[f"dW{l}"] = (1.0/m) * np.dot(dZ, A_prev.T)
        grads[f"db{l}"] = (1.0/m) * np.sum(dZ, axis=1, keepdims=True)
        dA = np.dot(W.T, dZ)
    return grads

# --- Source task: 3-class price prediction --  trained WITHOUT the airconditioning feature, so its
# input space exactly matches the downstream task below (transfer requires matching input dimensions).
ac_col = feature_cols.index("airconditioning")
X_src_all = np.delete(X_all, ac_col, axis=1)
mu_src = X_src_all[tr_idx].mean(axis=0, keepdims=True)
sigma_src = X_src_all[tr_idx].std(axis=0, keepdims=True) + 1e-8
X_src_train = ((X_src_all[tr_idx] - mu_src) / sigma_src).T

q1, q2 = df["price"].quantile([1/3, 2/3])
price_class_all = pd.cut(df["price"], bins=[-np.inf, q1, q2, np.inf], labels=[0,1,2]).astype(int).values
Y_class_train = one_hot(price_class_all[tr_idx], 3)

source_dims = [X_src_train.shape[0], 10, 5, 3]
params_source = initialize_parameters_deep(source_dims, seed=1)
for it in range(1500):
    AL, caches = L_model_forward_softmax(X_src_train, params_source)
    grads = L_model_backward_softmax(AL, Y_class_train, caches)
    params_source = update_parameters_gd(params_source, grads, alpha=0.1)

print("Source task (3-class price, trained without the airconditioning feature) trained.")
print("Layer 1 & 2 weights will be TRANSFERRED. W1 shape:", params_source["W1"].shape,
      " W2 shape:", params_source["W2"].shape)

Source task (3-class price, trained without the airconditioning feature) trained.
Layer 1 & 2 weights will be TRANSFERRED. W1 shape: (10, 12)  W2 shape: (5, 10)


In [17]:
# --- Downstream (target) task: predict 'airconditioning', with only a SMALL amount of labeled data ---
ac_col = feature_cols.index("airconditioning")
X_ac_all = np.delete(X_all, ac_col, axis=1)   # remove airconditioning from the INPUT features
Y_ac_all = X_all[:, ac_col:ac_col+1]          # airconditioning becomes the LABEL instead

mu_ac = X_ac_all[tr_idx].mean(axis=0, keepdims=True)
sigma_ac = X_ac_all[tr_idx].std(axis=0, keepdims=True) + 1e-8
X_ac_train_full = ((X_ac_all[tr_idx] - mu_ac) / sigma_ac).T
Y_ac_train_full = Y_ac_all[tr_idx].T
X_ac_dev = ((X_ac_all[dv_idx] - mu_ac) / sigma_ac).T
Y_ac_dev = Y_ac_all[dv_idx].T

# Simulate scarce downstream labels: only 20 examples available for the new task
n_scarce = 20
X_ac_train, Y_ac_train = X_ac_train_full[:, :n_scarce], Y_ac_train_full[:, :n_scarce]
print(f"Downstream task 'predict airconditioning': only {n_scarce} labeled examples available")

def transfer_forward(X, W1, b1, W2, b2_new, W_out_new, b_out_new):
    """Uses TRANSFERRED (frozen) W1,b1 and W2 (relu), but a FRESH final layer for the new task."""
    A1 = relu(np.dot(W1, X) + b1)
    A2 = relu(np.dot(W2, A1) + b2_new)
    Z_out = np.dot(W_out_new, A2) + b_out_new
    A_out = sigmoid(Z_out)
    return A_out, (X, A1, A2)

# TRANSFER: reuse layer-1 weights from the source task (frozen), fine-tune everything from layer 2 onward
W1_frozen = params_source["W1"].copy()
b1_frozen = params_source["b1"].copy()

def train_transfer(X, Y, W1_frozen, b1_frozen, n_h2=5, num_iterations=800, alpha=0.3, seed=1):
    rng = np.random.RandomState(seed)
    W2 = rng.randn(n_h2, W1_frozen.shape[0]) * np.sqrt(2.0/W1_frozen.shape[0])
    b2 = np.zeros((n_h2, 1))
    W_out = rng.randn(1, n_h2) * np.sqrt(2.0/n_h2)
    b_out = np.zeros((1, 1))

    for it in range(num_iterations):
        A1 = relu(np.dot(W1_frozen, X) + b1_frozen)     # FROZEN -- not updated
        Z2 = np.dot(W2, A1) + b2
        A2 = relu(Z2)
        Z_out = np.dot(W_out, A2) + b_out
        A_out = sigmoid(Z_out)

        m = X.shape[1]
        dZ_out = A_out - Y
        dW_out = (1.0/m) * np.dot(dZ_out, A2.T)
        db_out = (1.0/m) * np.sum(dZ_out, axis=1, keepdims=True)
        dA2 = np.dot(W_out.T, dZ_out)
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = (1.0/m) * np.dot(dZ2, A1.T)
        db2 = (1.0/m) * np.sum(dZ2, axis=1, keepdims=True)

        W_out -= alpha * dW_out; b_out -= alpha * db_out
        W2 -= alpha * dW2; b2 -= alpha * db2
    return W2, b2, W_out, b_out

W2_t, b2_t, Wout_t, bout_t = train_transfer(X_ac_train, Y_ac_train, W1_frozen, b1_frozen)
preds_transfer, _ = transfer_forward(X_ac_dev, W1_frozen, b1_frozen, W2_t, b2_t, Wout_t, bout_t)
acc_transfer = np.mean((preds_transfer > 0.5) == Y_ac_dev) * 100

# FROM SCRATCH: same architecture, but layer 1 also trained fresh on the same 20 scarce examples
scratch_dims = [X_ac_train.shape[0], W1_frozen.shape[0], 5, 1]
params_scratch, _ = train_basic(X_ac_train, Y_ac_train, scratch_dims, num_iterations=800, alpha=0.3)
acc_scratch = np.mean(predict_deep(params_scratch, X_ac_dev) == Y_ac_dev) * 100

print(f"\nFrom scratch (only {n_scarce} examples, no transfer): dev accuracy = {acc_scratch:.1f}%")
print(f"Transfer learning (same {n_scarce} examples, layer 1 reused): dev accuracy = {acc_transfer:.1f}%")

Downstream task 'predict airconditioning': only 20 labeled examples available

From scratch (only 20 examples, no transfer): dev accuracy = 57.8%
Transfer learning (same 20 examples, layer 1 reused): dev accuracy = 61.5%


## 7. Multitask Learning

Instead of training one network per task, train **one** network with **multiple output units**, one per task, sharing all the earlier layers. This helps when the tasks share useful low-level features and no single task alone has tons of data — the shared layers effectively get trained on the *combined* data from every task.

Below: train one network to simultaneously predict **three** binary labels (`expensive`, `airconditioning`, `prefarea`) from the same shared hidden layers, versus three completely separate single-task networks, and compare — especially for whichever task has the weakest individual signal.

In [18]:
ac_idx, pref_idx = feature_cols.index("airconditioning"), feature_cols.index("prefarea")
multi_cols = [c for c in range(n_x) if c not in (ac_idx, pref_idx)]   # drop these 2 from INPUT (they're now labels)

X_multi_all = X_all[:, multi_cols]
mu_m = X_multi_all[tr_idx].mean(axis=0, keepdims=True)
sigma_m = X_multi_all[tr_idx].std(axis=0, keepdims=True) + 1e-8
X_multi_train = ((X_multi_all[tr_idx] - mu_m) / sigma_m).T
X_multi_dev   = ((X_multi_all[dv_idx] - mu_m) / sigma_m).T

Y_multi_train = np.vstack([Y_all[tr_idx].ravel(), X_all[tr_idx, ac_idx], X_all[tr_idx, pref_idx]])   # (3, m)
Y_multi_dev   = np.vstack([Y_all[dv_idx].ravel(), X_all[dv_idx, ac_idx], X_all[dv_idx, pref_idx]])

def train_multitask(X, Y, layer_dims, num_iterations=1500, alpha=0.2, seed=1):
    """Y has shape (num_tasks, m). Output layer has num_tasks sigmoid units, one per task."""
    params = initialize_parameters_deep(layer_dims, seed=seed)
    for it in range(num_iterations):
        AL, caches = L_model_forward(X, params)   # AL shape (num_tasks, m) -- multi-unit sigmoid output
        grads = L_model_backward(AL, Y, caches)    # the same dZ=A-Y shortcut works per-unit, independently
        params = update_parameters_gd(params, grads, alpha)
    return params

multi_dims = [X_multi_train.shape[0], 10, 5, 3]   # 3 output units, one per task
params_multi = train_multitask(X_multi_train, Y_multi_train, multi_dims)
AL_multi, _ = L_model_forward(X_multi_dev, params_multi)
preds_multi = (AL_multi > 0.5).astype(int)

task_names = ["expensive", "airconditioning", "prefarea"]
print(f"{'task':>16} | {'multitask acc':>14} | {'single-task acc':>16}")
print("-" * 52)
for i, name in enumerate(task_names):
    acc_multi = np.mean(preds_multi[i] == Y_multi_dev[i]) * 100
    params_single, _ = train_basic(X_multi_train, Y_multi_train[i:i+1, :],
                                    [X_multi_train.shape[0], 10, 5, 1], num_iterations=1500, alpha=0.2)
    acc_single = np.mean(predict_deep(params_single, X_multi_dev) == Y_multi_dev[i:i+1, :]) * 100
    print(f"{name:>16} | {acc_multi:>13.1f}% | {acc_single:>15.1f}%")

            task |  multitask acc |  single-task acc
----------------------------------------------------
       expensive |          81.7% |            77.1%
 airconditioning |          70.6% |            70.6%
        prefarea |          76.1% |            69.7%


## 8. What Is End-To-End Deep Learning?

Traditionally, complex tasks were solved with a **pipeline** of separate, hand-designed stages, each with its own intermediate representation (e.g. speech recognition: audio -> phonemes -> words -> transcript). **End-to-end** deep learning instead trains a single network directly from raw input to final output, letting the network discover whatever intermediate representation is actually useful, with no hand-engineered stages in between.

As a toy analogy on housing data: predicting `expensive` **directly** from raw features (end-to-end) versus a **2-stage pipeline** — first hand-engineer an interpretable "quality score" (a weighted combination of a few features, as in section 16 of an earlier notebook), then threshold that score to classify.

In [19]:
# Stage 1 of the pipeline: a hand-engineered "quality score" (same weighted heuristic idea as before)
quality_feats = ["area", "bathrooms", "stories", "parking", "airconditioning", "prefarea"]
quality_idx = [feature_cols.index(f) for f in quality_feats]
quality_weights = np.array([0.54, 0.52, 0.42, 0.38, 0.45, 0.33])

X_q_z = (X_all[:, quality_idx] - X_all[:, quality_idx].mean(axis=0)) / X_all[:, quality_idx].std(axis=0)
quality_score = X_q_z @ quality_weights

def pipeline_predict(idx_set, threshold):
    return (quality_score[idx_set] > threshold).astype(int)

pipeline_threshold = np.median(quality_score[tr_idx])
pipeline_dev_acc = np.mean(pipeline_predict(dv_idx, pipeline_threshold) == Y_all[dv_idx].ravel()) * 100

# End-to-end: a plain neural network trained directly on raw features -> label, no hand-engineered stage
e2e_params, _ = train_basic(X_train, Y_train, [n_x, 10, 5, 1], num_iterations=1500, alpha=0.2)
e2e_dev_acc = np.mean(predict_deep(e2e_params, X_dev) == Y_dev) * 100

print(f"2-stage pipeline (hand-engineered quality score -> threshold): dev accuracy = {pipeline_dev_acc:.1f}%")
print(f"End-to-end (raw features -> label, no hand-engineered stage):  dev accuracy = {e2e_dev_acc:.1f}%")

2-stage pipeline (hand-engineered quality score -> threshold): dev accuracy = 83.5%
End-to-end (raw features -> label, no hand-engineered stage):  dev accuracy = 75.2%


## 9. Whether To Use End-To-End Deep Learning

**Pros of end-to-end:** lets the data speak for itself rather than being constrained by human-designed intermediate representations (which might discard useful signal); requires designing less hand-crafted machinery.

**Cons of end-to-end:** typically needs **much more data** to learn a good mapping directly (the pipeline's hand-designed stages inject useful prior knowledge, which substitutes for data); may discard genuinely useful hand-engineered components; can be harder to debug when it fails (a pipeline's separate stages can each be inspected individually).

**Key question to ask:** is there enough data to learn the complete, genuinely complex $X \to Y$ mapping directly? Below: repeat the end-to-end vs. pipeline comparison at different **training-set sizes**, to see how the end-to-end approach's advantage depends on how much data is available — the central practical consideration in this decision.

In [20]:
print(f"{'training examples':>18} | {'pipeline dev acc':>17} | {'end-to-end dev acc':>19}")
print("-" * 60)
for n_examples in [10, 30, 80, 327]:
    X_sub, Y_sub = X_train[:, :n_examples], Y_train[:, :n_examples]
    e2e_p, _ = train_basic(X_sub, Y_sub, [n_x, 10, 5, 1], num_iterations=1500, alpha=0.2)
    e2e_acc = np.mean(predict_deep(e2e_p, X_dev) == Y_dev) * 100
    # (the pipeline's quality score/threshold don't depend on n_examples -- it's fixed, hand-engineered)
    print(f"{n_examples:>18} | {pipeline_dev_acc:>16.1f}% | {e2e_acc:>18.1f}%")

print("\n-> Here the pipeline stays ahead even with the FULL training set -- on a small, simple tabular")
print("   problem like this, a well-designed hand-engineered score is hard for a from-scratch network to")
print("   beat with so few examples. The general principle still holds (end-to-end needs enough data to")
print("   learn what hand-engineering supplies for free) -- it's just that 327 examples isn't 'enough'")
print("   here. This is itself the right way to make the section 9 decision: check empirically which")
print("   approach wins on YOUR data and data budget, rather than assuming end-to-end always catches up.")

 training examples |  pipeline dev acc |  end-to-end dev acc
------------------------------------------------------------
                10 |             83.5% |               77.1%
                30 |             83.5% |               79.8%
                80 |             83.5% |               76.1%
               327 |             83.5% |               75.2%

-> Here the pipeline stays ahead even with the FULL training set -- on a small, simple tabular
   problem like this, a well-designed hand-engineered score is hard for a from-scratch network to
   beat with so few examples. The general principle still holds (end-to-end needs enough data to
   learn what hand-engineering supplies for free) -- it's just that 327 examples isn't 'enough'
   here. This is itself the right way to make the section 9 decision: check empirically which
   approach wins on YOUR data and data budget, rather than assuming end-to-end always catches up.


## Summary

| Concept | Key idea |
|---|---|
| Cleaning incorrect labels | Deep learning tolerates random label noise reasonably well; systematic noise teaches a genuinely wrong, localized pattern |
| Build first system, then iterate | Ship a simple baseline fast; use bias/variance + error analysis to guide what's actually worth building next |
| Train/test on different distributions | Dev/test should be pure target-distribution data, even if train is a larger mixed/different-distribution set |
| Bias/variance with mismatched data | Add a train-dev set to separate variance (train-dev gap) from data mismatch (dev vs train-dev gap) |
| Addressing data mismatch | Study the difference manually; synthesize data carefully -- synthesis must preserve a correct X-Y relationship |
| Transfer learning | Reuse early layers from a data-rich source task for a data-scarce target task |
| Multitask learning | One shared network, multiple output units -- helps when tasks share structure and each has limited data |
| End-to-end deep learning | Learn input->output directly, no hand-designed intermediate stages |
| Whether to use it | Needs enough data to learn what a hand-engineered pipeline would otherwise supply for free |

Together with the earlier ML-strategy notebook, this covers the full "what to do when the model isn't good enough yet" toolkit -- the diagnostic and decision-making layer that sits on top of the training techniques from the other notebooks.